# SPY Risk Alert Project Pipeline

This cumulative pipeline covers Stage04 ingestion, Stage05 storage, Stage06 preprocessing, Stage07 outlier analysis, Stage08 EDA, and Stage09 feature engineering. It defaults to retained raw data; set `REFRESH_RAW = True` only for an intentional new Nasdaq source snapshot.


## 1. Project Root, Configuration, and Imports

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')  # project/notebooks -> project
ROOT = Path.cwd()
if not (ROOT / 'src' / 'ingestion.py').is_file():
    for candidate in (ROOT, *ROOT.parents):
        project_candidate = candidate / 'project'
        if (project_candidate / 'src' / 'ingestion.py').is_file():
            ROOT = project_candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date

import pandas as pd

from src.cleaning import clean_spy_ohlcv, write_cleaning_report
from src.eda import eda_summary, prepare_spy_eda_frame
from src.features import build_spy_features, feature_target_correlation
from src.modeling import run_baseline
from src.evaluation import bootstrap_pr_auc, sensitivity_tables
from src.reporting import build_stakeholder_report
from src.run_step import run_stakeholder_report
from src.productization import save_model_artifact
from src.monitoring import MONITORING_CONTRACT, validate_monitoring_contract
from src.outliers import (
    analyze_daily_return_outliers,
    summarize_return_sensitivity,
    write_outlier_report,
)
from src.config import get_processed_data_dir, get_raw_data_dir, load_env
from src.ingestion import (
    fetch_nasdaq_history, timestamp_utc, validate_spy_history, write_manifest, write_raw_csv
)
from src.storage import get_parquet_engine, read_df, validate_roundtrip, write_df

environment_loaded = load_env()
RAW_DIR = get_raw_data_dir()
PROCESSED_DIR = get_processed_data_dir()
print('working from:', ROOT.name)
print('Environment loaded:', environment_loaded)
print('Raw directory:', RAW_DIR)
print('Processed directory:', PROCESSED_DIR)

working from: project
Environment loaded: True
Raw directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/raw
Processed directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/processed


## 2. Stage04 Raw Snapshot

Normal runs reuse the latest retained raw CSV. An explicit refresh validates and saves a new raw snapshot plus manifest.

In [2]:
REFRESH_RAW = False
SYMBOL = 'SPY'
csv_schema = {
    'open': 'float64', 'high': 'float64', 'low': 'float64',
    'close': 'float64', 'volume': 'int64',
}
raw_candidates = sorted(RAW_DIR.glob('api_nasdaq_spy_daily_*.csv'))

if REFRESH_RAW:
    end_date = date.today()
    start_date = end_date.replace(year=end_date.year - 10)
    spy_raw, source_metadata = fetch_nasdaq_history(
        SYMBOL, start_date=start_date.isoformat(), end_date=end_date.isoformat()
    )
    validation = validate_spy_history(spy_raw)
    snapshot_timestamp = timestamp_utc()
    raw_path = write_raw_csv(
        spy_raw, RAW_DIR, 'api_nasdaq_spy_daily', timestamp=snapshot_timestamp
    )
    manifest_path = write_manifest(
        {
            'path': raw_path, 'dataset': 'SPY daily unadjusted OHLCV',
            'rows': len(spy_raw), 'columns': list(spy_raw.columns),
            'source_metadata': source_metadata, 'validation': validation,
        },
        RAW_DIR / f'ingestion_manifest_{snapshot_timestamp}.json',
    )
    print('Refreshed raw snapshot:', raw_path.name)
    print('Saved manifest:', manifest_path.name)
else:
    if not raw_candidates:
        raise FileNotFoundError('No Stage04 SPY raw snapshot found. Set REFRESH_RAW = True.')
    raw_path = raw_candidates[-1]
    snapshot_timestamp = raw_path.stem.removeprefix('api_nasdaq_spy_daily_')
    spy_raw = read_df(raw_path, parse_dates=['date'], dtype=csv_schema)
    validation = validate_spy_history(spy_raw)
    print('Reused raw snapshot:', raw_path.name)

print('Rows and columns:', validation['shape'])
print('Date range:', validation['date_min'], 'to', validation['date_max'])
spy_raw.head()

Reused raw snapshot: api_nasdaq_spy_daily_20260907-143336.csv
Rows and columns: [2512, 6]
Date range: 2016-09-07 to 2026-09-04


,date,open,high,low,close,volume
0,2016-09-07,218.84,219.2200,218.30,219.01,76302150
1,2016-09-08,218.62,218.9400,218.15,218.51,73855230
2,2016-09-09,216.97,217.0300,213.25,213.28,220309300
3,2016-09-12,212.39,216.8100,212.31,216.34,167653400
4,2016-09-13,214.84,215.1499,212.50,213.23,182323200


## 3. Stage05 Typed Storage

The Parquet file is a typed representation of the named raw snapshot, not a cleaning or feature step.

In [3]:
storage_path = PROCESSED_DIR / f'spy_ohlcv_nasdaq_{snapshot_timestamp}.parquet'
write_df(spy_raw, storage_path)
spy_stored = read_df(storage_path)
storage_roundtrip = validate_roundtrip(
    spy_raw, spy_stored,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not storage_roundtrip['passed']:
    raise ValueError(f'Stage05 storage validation failed: {storage_roundtrip}')
print('Parquet engine:', get_parquet_engine())
print('Stored file:', storage_path.name)
print('Storage round-trip passed:', storage_roundtrip['passed'])

Parquet engine: pyarrow
Stored file: spy_ohlcv_nasdaq_20260907-143336.parquet
Storage round-trip passed: True


## 4. Stage06 Deterministic Preprocessing

The policy reparses and validates canonical OHLCV fields, sorts dates, and records its non-actions. It does not impute prices, remove outliers, or fit a global scaler.

In [4]:
spy_clean, cleaning_report = clean_spy_ohlcv(spy_stored)
preprocessed_path = PROCESSED_DIR / f'spy_ohlcv_preprocessed_{snapshot_timestamp}.parquet'
report_path = PROCESSED_DIR / f'cleaning_report_{snapshot_timestamp}.json'
write_df(spy_clean, preprocessed_path)
write_cleaning_report(cleaning_report, report_path)
spy_reloaded = read_df(preprocessed_path)
preprocessing_roundtrip = validate_roundtrip(
    spy_clean, spy_reloaded,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not preprocessing_roundtrip['passed']:
    raise ValueError(f'Stage06 preprocessing validation failed: {preprocessing_roundtrip}')

print('Preprocessed file:', preprocessed_path.name)
print('Cleaning report:', report_path.name)
print('Rows dropped:', cleaning_report['rows_dropped'])
print('Preprocessing round-trip passed:', preprocessing_roundtrip['passed'])

Preprocessed file: spy_ohlcv_preprocessed_20260907-143336.parquet
Cleaning report: cleaning_report_20260907-143336.json
Rows dropped: 0
Preprocessing round-trip passed: True


## 5. Stage07 Return Outlier Analysis

Daily close-to-close returns are flagged with IQR (`k=1.5`) and z-score (`|z| > 3`) rules. All market dates are retained. Filtered and 5%/95% winsorized variants are sensitivity diagnostics, not production transformations.


In [5]:
spy_return_analysis, outlier_report = analyze_daily_return_outliers(spy_clean)
sensitivity_summary = summarize_return_sensitivity(spy_return_analysis)

flags_path = PROCESSED_DIR / f'spy_return_outlier_flags_{snapshot_timestamp}.parquet'
sensitivity_path = PROCESSED_DIR / f'return_outlier_sensitivity_{snapshot_timestamp}.csv'
outlier_report_path = PROCESSED_DIR / f'outlier_analysis_report_{snapshot_timestamp}.json'
write_df(spy_return_analysis, flags_path)
write_df(sensitivity_summary, sensitivity_path)
write_outlier_report(outlier_report, outlier_report_path)

flags_reloaded = read_df(flags_path)
sensitivity_reloaded = read_df(sensitivity_path)
flags_roundtrip = validate_roundtrip(
    spy_return_analysis,
    flags_reloaded,
    {'date': 'datetime', 'close': 'float', 'daily_return': 'float', 'volume': 'integer'},
)
sensitivity_roundtrip = validate_roundtrip(
    sensitivity_summary,
    sensitivity_reloaded,
    {'observations': 'integer', 'mean': 'float', 'std': 'float'},
)
if not flags_roundtrip['passed'] or not sensitivity_roundtrip['passed']:
    raise ValueError('Stage07 storage validation failed')

print('Outlier flags:', flags_path.name)
print('Sensitivity table:', sensitivity_path.name)
print('IQR flags retained:', outlier_report['iqr']['flagged_count'])
print('Z-score flags retained:', outlier_report['zscore']['flagged_count'])
print('Outlier round-trips passed:', flags_roundtrip['passed'] and sensitivity_roundtrip['passed'])
display(sensitivity_summary)


Outlier flags: spy_return_outlier_flags_20260907-143336.parquet
Sensitivity table: return_outlier_sensitivity_20260907-143336.csv
IQR flags retained: 165
Z-score flags retained: 33
Outlier round-trips passed: True


,variant,observations,mean,median,std,minimum,maximum
0,all,2511,0.000566,0.000699,0.011363,-0.109424,0.105019
1,filtered_iqr,2346,0.001043,0.000848,0.007344,-0.018158,0.019981
2,winsorized_0.05_0.95,2511,0.000642,0.000699,0.008169,-0.016792,0.015605


## 6. Stage08 Exploratory Data Analysis

This summary profiles retained Stage07 observations and EDA-only return/range/volume/rolling-volatility fields. It records descriptive patterns and attention flags without treating correlation as causation or full-sample values as deployment rules.


In [6]:
spy_eda = prepare_spy_eda_frame(spy_return_analysis)
eda_tables = eda_summary(spy_eda)
eda_paths = {
    name: PROCESSED_DIR / f"spy_eda_{name}_{snapshot_timestamp}.csv"
    for name in eda_tables
}
for name, table in eda_tables.items():
    write_df(table, eda_paths[name])

for name, table in eda_tables.items():
    reloaded = read_df(eda_paths[name])
    if len(reloaded) != len(table) or list(reloaded.columns) != list(table.columns):
        raise ValueError(f"Stage08 EDA table validation failed for {name}")

attention_records = len(eda_tables["attention"])
print("EDA tables:", ", ".join(path.name for path in eda_paths.values()))
print("EDA rows retained:", len(spy_eda))
print("Structural missing cells:", int(spy_eda.isna().sum().sum()))
print("Attention records:", attention_records)
display(eda_tables["attention"])


EDA tables: spy_eda_overview_20260907-143336.csv, spy_eda_column_profile_20260907-143336.csv, spy_eda_numeric_summary_20260907-143336.csv, spy_eda_categorical_summary_20260907-143336.csv, spy_eda_datetime_summary_20260907-143336.csv, spy_eda_attention_20260907-143336.csv
EDA rows retained: 2512
Structural missing cells: 23
Attention records: 4


,column,role,missing_count,missing_fraction,dominant_fraction,attention
0,daily_return,numeric,1,0.000398,0.001990,has_missing
1,return_outlier_zscore,categorical,0,0.000000,0.986863,dominant_category
2,abs_return,numeric,1,0.000398,0.001990,has_missing
3,rolling_volatility_21,numeric,21,0.008360,0.008360,has_missing


## 7. Stage09 Leakage-Aware Feature Engineering

Each `*_t` feature uses only the close and history available at date `t`. The shifted `next_day_abs_return` is outcome-only; no full-snapshot high-volatility threshold, imputation, scaling, or feature selection occurs here. Those decisions must use training history alone in later chronological modeling.


In [7]:
spy_features = build_spy_features(spy_return_analysis)
feature_correlations = feature_target_correlation(spy_features)

feature_path = PROCESSED_DIR / f"spy_feature_candidates_{snapshot_timestamp}.parquet"
correlation_path = PROCESSED_DIR / f"feature_target_correlations_{snapshot_timestamp}.csv"
write_df(spy_features, feature_path)
write_df(feature_correlations, correlation_path)

features_reloaded = read_df(feature_path)
correlations_reloaded = read_df(correlation_path)
features_roundtrip = validate_roundtrip(
    spy_features,
    features_reloaded,
    {
        "date": "datetime", "return_t": "float", "rolling_volatility_5_t": "float",
        "weekday_Monday": "integer", "next_day_abs_return": "float",
    },
)
if not features_roundtrip["passed"]:
    raise ValueError(f"Stage09 feature storage validation failed: {features_roundtrip}")
if list(correlations_reloaded.columns) != list(feature_correlations.columns):
    raise ValueError("Stage09 correlation-table columns changed on reload")
if len(spy_features) != len(spy_return_analysis):
    raise ValueError("Stage09 must retain one row per market date")
if not spy_features["date"].is_monotonic_increasing:
    raise ValueError("Stage09 feature dates must be sorted")
if int(spy_features["return_t"].isna().sum()) != 1:
    raise ValueError("Unexpected return warm-up missingness")
if int(spy_features["rolling_volatility_5_t"].isna().sum()) != 5:
    raise ValueError("Unexpected rolling-volatility warm-up missingness")
if int(spy_features["next_day_abs_return"].isna().sum()) != 1:
    raise ValueError("Unexpected terminal outcome missingness")
weekday_columns = [column for column in spy_features if column.startswith("weekday_")]
if weekday_columns != [
    "weekday_Monday", "weekday_Tuesday", "weekday_Wednesday", "weekday_Thursday", "weekday_Friday"
]:
    raise ValueError(f"Unexpected weekday encoding columns: {weekday_columns}")
if not spy_features[weekday_columns].sum(axis=1).eq(1).all():
    raise ValueError("Weekday one-hot encoding must contain exactly one active field per row")

print("Feature table:", feature_path.name)
print("Correlation diagnostic:", correlation_path.name)
print("Rows retained:", len(spy_features))
print("Feature candidates:", len(feature_correlations))
print("Feature round-trip passed:", features_roundtrip["passed"])
display(feature_correlations)


Feature table: spy_feature_candidates_20260907-143336.parquet
Correlation diagnostic: feature_target_correlations_20260907-143336.csv
Rows retained: 2512
Feature candidates: 11
Feature round-trip passed: True


,feature,observations,pearson_correlation
0,intraday_range_t,2511,0.506006
1,rolling_volatility_5_t,2506,0.498749
2,stress_interaction_t,2510,0.403360
3,abs_return_t,2510,0.357965
4,return_t,2510,-0.109967
5,weekday_Thursday,2511,0.028947
6,weekday_Monday,2511,-0.021912
7,log_volume_change_t,2510,0.011700
8,weekday_Tuesday,2511,-0.010849
9,weekday_Wednesday,2511,0.010547


## 8. Stage10 Time-Aware Classification Baseline

This stage uses a training-only event threshold and chronological train/validation/test blocks. Validation selects the regularization by PR-AUC and the cutoff by F1; test is reserved for a future-like report.


In [8]:
modeling = run_baseline(spy_features)
modeling_frame_path = PROCESSED_DIR / f"spy_modeling_frame_{snapshot_timestamp}.parquet"
validation_metrics_path = PROCESSED_DIR / f"model_validation_metrics_{snapshot_timestamp}.csv"
test_metrics_path = PROCESSED_DIR / f"model_test_metrics_{snapshot_timestamp}.csv"
write_df(modeling["data"], modeling_frame_path)
pd.DataFrame([modeling["validation_metrics"]]).to_csv(validation_metrics_path, index=False)
pd.DataFrame([modeling["test_metrics"]]).to_csv(test_metrics_path, index=False)
assert modeling["train"]["date"].max() < modeling["validation"]["date"].min() < modeling["test"]["date"].min()
print("Stage10 cutoff:", modeling["cutoff"], "test PR-AUC:", modeling["test_metrics"]["pr_auc"])


Stage10 cutoff: 0.5199999999999999 test PR-AUC: 0.29281211176015853


## 9. Stage11 Evaluation and Risk Communication

Bootstrap uncertainty, cutoff sensitivity, and volatility-regime diagnostics evaluate the fixed Stage10 future-test predictions without retuning them.


In [9]:
evaluation_ci,_=bootstrap_pr_auc(modeling["test"]["label"],modeling["test_probability"],n_boot=600,seed=111)
evaluation_scenarios,evaluation_subgroups=sensitivity_tables(modeling["test"],modeling["test_probability"],modeling["cutoff"])
pd.DataFrame([evaluation_ci]).to_csv(PROCESSED_DIR/f"bootstrap_pr_auc_{snapshot_timestamp}.csv",index=False)
evaluation_scenarios.to_csv(PROCESSED_DIR/f"evaluation_scenarios_{snapshot_timestamp}.csv",index=False)
evaluation_subgroups.to_csv(PROCESSED_DIR/f"evaluation_subgroups_{snapshot_timestamp}.csv",index=False)
assert evaluation_ci["lower_95"]<=evaluation_ci["upper_95"] and len(evaluation_scenarios)==2 and len(evaluation_subgroups)==2
print("Stage11 PR-AUC interval:",evaluation_ci["lower_95"],evaluation_ci["upper_95"])

Stage11 PR-AUC interval: 0.17913326866758658 0.4572122384144519


## Sources, Storage, Preprocessing, Outliers, EDA, Features, Assumptions, and Risks

- Source rules: `docs/data_sources.md`; storage lineage: `docs/data_storage.md`; preprocessing policy: `docs/preprocessing.md`; outlier policy: `docs/outliers.md`; EDA policy: `docs/eda.md`; feature definitions: `docs/feature_definitions.md`.
- `data/raw/` remains immutable; processed outputs are reproducible from the named raw snapshot and code.
- Stage07 flags and retains extreme returns. Stage08 documents full-snapshot descriptive patterns; Stage09 makes time-safe candidate features but does not define a production event threshold or make causal claims.
- Train-only target thresholds, chronological model validation, and evaluation are documented above. Stage12 generates a decision-oriented report from fixed outputs; it does not refit or retune the model.


## 10. Stage12 Stakeholder Delivery

Generate a decision-oriented report from the fixed Stage10/11 outputs. The recommendation is a human review trigger only; no model fitting, cutoff selection, or test retuning occurs in this step.

In [10]:
stakeholder_report_path = build_stakeholder_report(
    test_metrics_path,
    PROCESSED_DIR / f"bootstrap_pr_auc_{snapshot_timestamp}.csv",
    PROCESSED_DIR / f"evaluation_scenarios_{snapshot_timestamp}.csv",
    PROCESSED_DIR / f"evaluation_subgroups_{snapshot_timestamp}.csv",
    f"spy_evaluation_{snapshot_timestamp}.png",
    ROOT / "reports" / "spy_risk_alert_stakeholder_report.md",
)
assert stakeholder_report_path.is_file()
print("Stage12 stakeholder report:", stakeholder_report_path)

Stage12 stakeholder report: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/reports/spy_risk_alert_stakeholder_report.md


## 11. Stage13 Productization

Persist the selected Stage10 pipeline together with its feature schema and fixed decision metadata. Saving this artifact does not train a new model; it makes the already fitted baseline reusable by the local API.

In [11]:
model_artifact_path = save_model_artifact(
    modeling["model"],
    modeling["feature_columns"],
    modeling["cutoff"],
    modeling["threshold"],
    str(modeling["train"]["date"].max().date()),
    ROOT / "model" / "spy_risk_alert_model.pkl",
)
assert model_artifact_path.is_file()
print("Stage13 saved model:", model_artifact_path)

Stage13 saved model: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/model/spy_risk_alert_model.pkl


## 12. Stage14 Deployment and Monitoring

Validate that the version-controlled monitoring contract covers Data, Model, System, and Business layers. The metrics are a conceptual runbook design; this pipeline does not create production telemetry or send alerts.

In [12]:
monitoring_docs = [ROOT / "docs" / "monitoring_plan.md", ROOT / "docs" / "handoff_plan.md"]
assert all(path.is_file() for path in monitoring_docs)
monitoring_coverage = validate_monitoring_contract(MONITORING_CONTRACT)
assert {"Business", "Data", "Model", "System"}.issubset(monitoring_coverage)
print("Stage14 monitoring coverage:", monitoring_coverage)
print("Stage14 handoff docs:", ", ".join(path.name for path in monitoring_docs))

Stage14 monitoring coverage: {'Business': 1, 'Data': 2, 'Model': 1, 'System': 1}
Stage14 handoff docs: monitoring_plan.md, handoff_plan.md


## 13. Stage15 Orchestration and System Design

Re-run the final report through its CLI-ready reusable function. This validates the persisted Stage10/11 checkpoints, logs a stage boundary, and produces the same idempotent stakeholder report without refitting the model.

In [13]:
orchestrated_report_path = run_stakeholder_report(snapshot_timestamp)
assert orchestrated_report_path == ROOT / "reports" / "spy_risk_alert_stakeholder_report.md"
assert orchestrated_report_path.is_file()
print("Stage15 orchestrated report:", orchestrated_report_path)

2026-09-08 18:49:31,201 | INFO | stage=stakeholder_report start timestamp=20260907-143336


2026-09-08 18:49:31,204 | INFO | stage=stakeholder_report checkpoint=/Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/reports/spy_risk_alert_stakeholder_report.md


Stage15 orchestrated report: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/reports/spy_risk_alert_stakeholder_report.md


## 14. Stage16 Lifecycle Review

Verify that the final repository contains the required lifecycle guide, nontechnical summary, operational documentation, and core project folders. This is a documentation and completeness check; it does not train or retune a model.

In [14]:
final_docs = [
    ROOT / "README.md",
    ROOT / "docs" / "lifecycle_framework_guide.md",
    ROOT / "docs" / "project_summary.md",
    ROOT / "docs" / "monitoring_plan.md",
    ROOT / "docs" / "orchestration_plan.md",
]
final_dirs = [ROOT / name for name in ("data/raw", "data/processed", "notebooks", "src", "reports", "model", "docs")]
assert all(path.is_file() for path in final_docs)
assert all(path.is_dir() for path in final_dirs)
lifecycle_guide = (ROOT / "docs" / "lifecycle_framework_guide.md").read_text(encoding="utf-8")
assert all(f"Stage {stage:02d}" in lifecycle_guide for stage in range(17))
print("Stage16 final docs:", ", ".join(path.name for path in final_docs))
print("Stage16 required directories:", ", ".join(path.name for path in final_dirs))

Stage16 final docs: README.md, lifecycle_framework_guide.md, project_summary.md, monitoring_plan.md, orchestration_plan.md
Stage16 required directories: raw, processed, notebooks, src, reports, model, docs
